<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 13th exercise: <font color="#C70039">First Reinforcement Learning Q-Table learning</font>
* Course: AML
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date:   01.08.2026

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e0/Q-Learning_Matrix_Initialized_and_After_Training.png" style="float: center;" width="450">

---------------------------------
**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="FFC300">TASKS</font>:
The tasks that you need to work on within this notebook are always indicated below as bullet points. 
If a task is more challenging and consists of several steps, this is indicated as well. 
Make sure you have worked down the task list and commented your doings. 
This should be done by using markdown.<br> 
<font color=red>Make sure you don't forget to specify your name and your matriculation number in the notebook.</font>

**YOUR TASKS in this exercise are as follows**:
1. import the notebook to Google Colab or use your local machine.
2. make sure you specified your name and your matriculation number in the header below my name and date. 
    * set the date too and remove mine.
3. read the entire notebook carefully 
    * add comments wherever you feel it necessary for better understanding
    * run the notebook for the first time. 
4. play with all hyperparameters including the actions, states, rewards table.
5. add and implement an ϵ-greedy strategy 
---------------------------------

In [ ]:
# Google Colab setup: make repository files available under the expected relative paths.
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    repository = "/content/AML"
    if not os.path.isdir(repository):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/gheisenberg/AML.git", repository], check=True)
    os.chdir(repository)

In [ ]:
# NumPy stores the reward graph and Q-values.
# The random generator controls reproducible exploration.
import numpy as np

rng = np.random.default_rng(1)

### Create the possible states

In [ ]:
# Convert readable location labels into numeric indexes for the Q-table.
# The reverse mapping turns the learned route back into location names.
location_to_state = {
    "L1": 0,
    "L2": 1,
    "L3": 2,
    "L4": 3,
    "L5": 4,
    "L6": 5,
    "L7": 6,
    "L8": 7,
    "L9": 8,
}

state_to_location = {state: location for location, state in location_to_state.items()}

### Create the actions & rewards

In [ ]:
# Actions represent candidate transitions to other location states.
actions = list(range(len(location_to_state)))

In [ ]:
# Positive entries define permitted moves; zero means no direct connection.
# The reward matrix is the environment from which the agent learns.
rewards = np.array([[0,1,0,0,0,0,0,0,0],
                   [1,0,1,0,1,0,0,0,0],
                   [0,1,0,0,0,1,0,0,0],
                   [0,0,0,0,0,0,1,0,0],
                   [0,1,0,0,0,0,0,1,0],
                   [0,0,1,0,0,0,0,0,0],
                   [0,0,0,1,0,0,0,1,0],
                   [0,0,0,0,1,0,1,0,1],
                   [0,0,0,0,0,0,0,1,0]])

### Define the remaining hyperparameters

In [ ]:
# Alpha controls update size, gamma values future rewards, and epsilon drives exploration.
# Change these parameters and compare the learned route.
gamma = 0.99  # Discount factor
alpha = 0.7  # Learning rate
epsilon = 1.0  # Initial exploration probability
epsilon_min = 0.05
epsilon_decay = 0.995

### Define agent and its attributes

In [ ]:
# The agent learns values from the graph without being told the optimal route.
# Epsilon-greedy selection explores reachable actions but increasingly exploits known values.
class QAgent:
    def __init__(
        self,
        alpha,
        gamma,
        epsilon,
        epsilon_min,
        epsilon_decay,
        location_to_state,
        actions,
        rewards,
        state_to_location,
        random_generator,
    ):
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.location_to_state = location_to_state
        self.actions = actions
        self.rewards = rewards
        self.state_to_location = state_to_location
        self.rng = random_generator

        number_of_states = len(location_to_state)
        self.q_table = np.zeros((number_of_states, number_of_states), dtype=float)

    def training(self, start_location, end_location, iterations):
        rewards_with_goal = self.rewards.astype(float).copy()
        goal_state = self.location_to_state[end_location]
        rewards_with_goal[goal_state, goal_state] = 100.0

        number_of_states = len(self.location_to_state)
        for _ in range(iterations):
            current_state = int(self.rng.integers(number_of_states))
            playable_actions = np.flatnonzero(rewards_with_goal[current_state] > 0)
            if playable_actions.size == 0:
                continue

            # Epsilon-greedy action selection restricted to reachable states.
            if self.rng.random() < self.epsilon:
                next_state = int(self.rng.choice(playable_actions))
            else:
                playable_values = self.q_table[current_state, playable_actions]
                next_state = int(playable_actions[np.argmax(playable_values)])

            best_future_value = self.q_table[next_state].max()
            temporal_difference = (
                rewards_with_goal[current_state, next_state]
                + self.gamma * best_future_value
                - self.q_table[current_state, next_state]
            )
            self.q_table[current_state, next_state] += self.alpha * temporal_difference
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

        route = self.get_optimal_route(start_location, end_location)
        print("Q-table:")
        print(np.round(self.q_table, 2))
        print("Optimal route:", route)
        return route

    def get_optimal_route(self, start_location, end_location):
        route = [start_location]
        current_location = start_location
        maximum_steps = len(self.location_to_state) * 2

        for _ in range(maximum_steps):
            if current_location == end_location:
                return route

            current_state = self.location_to_state[current_location]
            playable_actions = np.flatnonzero(self.rewards[current_state] > 0)
            if playable_actions.size == 0:
                raise RuntimeError(f"No route continues from {current_location}.")

            playable_values = self.q_table[current_state, playable_actions]
            next_state = int(playable_actions[np.argmax(playable_values)])
            current_location = self.state_to_location[next_state]
            route.append(current_location)

        raise RuntimeError("The learned policy entered a cycle before reaching the goal.")

In [ ]:
# Train from L9 to L4 and print the route derived from the learned Q-table.
q_agent = QAgent(
    alpha=alpha,
    gamma=gamma,
    epsilon=epsilon,
    epsilon_min=epsilon_min,
    epsilon_decay=epsilon_decay,
    location_to_state=location_to_state,
    actions=actions,
    rewards=rewards,
    state_to_location=state_to_location,
    random_generator=rng,
)
q_agent.training("L9", "L4", iterations=1_000)